# NB04A — Three-year quarterly Sentinel-2 audit and export

Restartable four-site seasonal composite extraction. Run sequentially in Colab. **Do not run the NB04 benchmark.** Outputs are UAV-year-aligned, not necessarily pre-flight.

**Corrected export:** Earlier rasters had width/height swapped at download. Their output folder is retained as historical, but should not be used for modelling. Corrected data go to `04A_quarterly_3year_gridfix`.

With geedim, projected image footprints may differ from canonical raster origins by a fraction of one pixel. The export accepts matching dimensions and ≤0.5 pixel maximum discrepancy, logs the transforms, then resamples explicitly to the canonical grid. Larger mismatches fail.

In [ ]:
# 00 — EDIT SETTINGS HERE ONLY
from pathlib import Path
ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Musanga")
PROJECT = "musanga-511110"
SITES = {"Baego": (2021,2023), "MOS": (2022,2024), "Yangambi": (2022,2024), "Yoko": (2018,2020)}
ANNUAL_DIR = ROOT / "Final/Outputs/V3/04_satellite_Musanga/features/RAW_ANNUAL"
OUTPUT_DIR = ROOT / "Final/Outputs/V3_1/04A_quarterly_3year_gridfix"  # previous export had swapped dimensions
ROI_PATH = ROOT / "data/Orthoo_RGB/ROI.shp"  # full mosaic unchanged; used for separate ROI QA
MIN_CLEAR_OBSERVATIONS = 1
EXPORT_SCALE_M = 10
OVERWRITE_VERIFIED = False  # Deliberately never used to overwrite valid output


In [ ]:
# 01 — Mount, dependencies, and initialization
import os, json, shutil, tempfile, traceback
import numpy as np, pandas as pd, rasterio
from rasterio.features import geometry_mask
from rasterio.warp import transform_geom
from google.colab import drive
drive.mount("/content/drive")
import subprocess, sys
subprocess.check_call([sys.executable,"-m","pip","-q","install","earthengine-api","geemap","geedim","geopandas"])
import ee, geemap, geedim, geopandas as gpd
print("geedim ready:", geedim.__file__,flush=True)
ee.Authenticate()
ee.Initialize(project=PROJECT)
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
if not ROI_PATH.exists():
    manifest=ROOT / "Final/Outputs/V3/00_input_audit/00_input_manifest.json"
    if manifest.exists():
        ROI_PATH=Path(json.loads(manifest.read_text())["roi"])
assert ROI_PATH.exists(),f"ROI shapefile missing: {ROI_PATH}"
BANDS=["B2","B3","B4","B8","B5","B6","B7","B8A","B11","B12"]
SCL_BAD=[0,1,3,7,8,9,10,11]
print("Initialized", PROJECT,flush=True)


In [ ]:
# 02 — Resolve canonical annual grids, inspect existing seasonal files
def grid_for(site,year):
    matches=sorted(ANNUAL_DIR.rglob(f"{site}_RAW_ANNUAL_{year}_10m.tif"))
    if len(matches)!=1: raise FileNotFoundError(f"{site}: expected exactly one annual grid, found {matches}")
    with rasterio.open(matches[0]) as ds:
        if ds.crs is None: raise ValueError("Annual grid has no CRS")
        return matches[0],ds.profile.copy(),ds.bounds
def output_for(site,start,end):
    return OUTPUT_DIR / f"{site}_S2_QUARTERLY_3YR_{start}_{end}_10m.tif"
for site,(start,end) in SITES.items():
    try:
        p,prof,bounds=grid_for(site,end)
        print(site,"GRID",p,"CRS",prof["crs"],"size",prof["width"],prof["height"],
              "EXISTING",output_for(site,start,end).exists(),flush=True)
    except Exception as e: print(site,"AUDIT ERROR",repr(e),flush=True)


In [ ]:
# 03 — Earth Engine: pooled observations, quarter medians and counts
def mask_s2(img):
    good=img.select("SCL").remap(SCL_BAD,[0]*len(SCL_BAD),1).eq(1)
    return img.select(BANDS).updateMask(good).multiply(0.0001).copyProperties(img,["system:time_start"])
def make_stack(start,end,region):
    col=(ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
         .filterBounds(region).filterDate(f"{start}-01-01",f"{end+1}-01-01")
         .map(mask_s2))
    parts=[]
    for q in range(4):
        months=[q*3+1,q*3+2,q*3+3]
        quarter=col.filter(ee.Filter.calendarRange(months[0],months[-1],"month"))
        med=quarter.median().select(BANDS).rename([f"Q{q+1}_{b}" for b in BANDS])
        cnt=quarter.select("B2").count().rename(f"Q{q+1}_nclear").toFloat()
        parts.extend([med.toFloat(),cnt])
    return ee.Image.cat(parts).toFloat()


In [ ]:
# 04 — Verified-file QA (44 bands, grid alignment, quarterly coverage, ROI)
def audit_raster(path,profile,roi_path=None):
    with rasterio.open(path) as ds:
        if ds.count!=44: raise ValueError(f"Expected 44 bands; got {ds.count}")
        if (ds.crs!=profile["crs"] or ds.transform!=profile["transform"] or
            ds.width!=profile["width"] or ds.height!=profile["height"]):
            raise ValueError("Raster does not match annual canonical grid")
        if ds.nodata is None: raise ValueError("Missing explicit NoData")
        roi_mask=None
        if roi_path:
            roi=gpd.read_file(roi_path)
            if roi.crs is None: raise ValueError("ROI CRS missing")
            roi=roi.to_crs(ds.crs)
            roi_mask=geometry_mask(roi.geometry,transform=ds.transform,
                                   out_shape=(ds.height,ds.width),invert=True)
            if not roi_mask.any(): raise ValueError("ROI has no overlap with grid")
        rows=[]
        for q in range(4):
            n=ds.read(q*11+11,masked=True)
            arr=np.ma.filled(n,np.nan)
            valid=np.isfinite(arr)&(arr>=MIN_CLEAR_OBSERVATIONS)
            reflect_valid=valid.copy()
            for b in range(1,11):
                a=ds.read(q*11+b,masked=True)
                reflect_valid &= np.isfinite(np.ma.filled(a,np.nan))
            if np.any(np.isfinite(arr)&(arr<0)): raise ValueError("Negative clear counts")
            rows.append(dict(quarter=q+1,grid_valid_fraction=float(reflect_valid.mean()),
                roi_valid_fraction=float(reflect_valid[roi_mask].mean()) if roi_mask is not None else np.nan,
                mean_clear_count=float(np.nanmean(arr[valid])) if valid.any() else 0,
                median_clear_count=float(np.nanmedian(arr[valid])) if valid.any() else 0,
                valid_pixels=int(reflect_valid.sum()),grid_pixels=arr.size))
        return rows


In [ ]:
# 05 — Export only absent or invalid files; SSD first, verify before copy
records=[]
for site,(start,end) in SITES.items():
    print("\nSITE",site,flush=True)
    try:
        annual,profile,bounds=grid_for(site,end)
        target=output_for(site,start,end)
        if target.exists():
            try:
                rows=audit_raster(target,profile,ROI_PATH)
                print("VERIFIED CACHE — skipping",target,flush=True)
                records += [dict(site=site,years=f"{start}-{end}",status="verified_cache",**r) for r in rows]
                continue
            except Exception as exc:
                print("INVALID EXISTING FILE — preserved, not overwritten:",repr(exc),flush=True)
                records.append(dict(site=site,years=f"{start}-{end}",status="invalid_existing",error=str(exc)))
                continue
        from rasterio.warp import transform_bounds
        west,south,east,north=transform_bounds(profile["crs"],"EPSG:4326",*bounds,densify_pts=21)
        region=ee.Geometry.Rectangle([west,south,east,north],proj="EPSG:4326",geodesic=False)
        stack=make_stack(start,end,region)
        with tempfile.TemporaryDirectory(prefix="musanga_s2_",dir="/content") as temp:
            raw=Path(temp)/"ee_raw.tif"
            aligned=Path(temp)/"aligned.tif"
            print("Downloading 44 bands to SSD",flush=True)
            geemap.download_ee_image(stack,filename=str(raw),region=region,
                crs=profile["crs"].to_string(),crs_transform=list(profile["transform"])[:6],
                shape=(profile["height"],profile["width"]))
            # The geedim tile merger may return slightly different bounds/shape.
            # Record source geometry and EXPLICITLY align bands to the existing
            # annual 10 m grid. This is never an implicit or silent resample.
            from rasterio.warp import reproject, Resampling, transform_bounds
            with rasterio.open(raw) as src:
                if src.count != 44: raise ValueError(f"Download incomplete: {src.count} bands")
                if src.crs is None: raise ValueError("Earth Engine download has no CRS")
                print("Downloaded:", src.crs, src.width, src.height, tuple(src.transform)[:6], flush=True)
                print("Canonical: ", profile["crs"], profile["width"], profile["height"],
                      tuple(profile["transform"])[:6], flush=True)
                same_grid = (src.crs == profile["crs"] and src.shape ==
                             (profile["height"], profile["width"]) and
                             src.transform.almost_equals(profile["transform"], precision=1e-7))
                if not same_grid:
                    # Require exact dimensions and <0.5-pixel discrepancy; perform explicit subpixel resampling.
                    px=abs(profile["transform"].a); py=abs(profile["transform"].e)
                    transform_error=max(abs(src.transform.c-profile["transform"].c)/px,
                                        abs(src.transform.f-profile["transform"].f)/py,
                                        abs(src.transform.a-profile["transform"].a)/px,
                                        abs(src.transform.e-profile["transform"].e)/py)
                    expected_shape=(profile["height"],profile["width"])
                    if src.shape!=expected_shape or transform_error>0.5:
                        raise ValueError(f"Export pixel geometry invalid: got {src.shape}, expected {expected_shape}, transform error {transform_error:.4f} pixels")
                    # Prevent grossly wrong alignment (e.g., different site/region).
                    sb = transform_bounds(src.crs,profile["crs"],*src.bounds,densify_pts=21)
                    db = rasterio.transform.array_bounds(profile["height"],profile["width"],profile["transform"])
                    overlap=max(0,min(sb[2],db[2])-max(sb[0],db[0]))*max(0,min(sb[3],db[3])-max(sb[1],db[1]))
                    fraction=overlap/max(1e-9,(db[2]-db[0])*(db[3]-db[1]))
                    print(f"EXPLICIT canonical-grid alignment; footprint overlap={fraction:.4f}",flush=True)
                    if fraction < 0.98: raise ValueError(f"Downloaded footprint differs substantially: {fraction:.3f}")
                p=profile.copy()
                for field in ("photometric","alpha","extrasamples","NBITS","colorinterp","interleave"):
                    p.pop(field,None)
                p.update(driver="GTiff",count=44,dtype="float32",nodata=-9999.0,
                         compress="deflate",predictor=3)
                with rasterio.open(aligned,"w",**p) as dst:
                    for k in range(1,45):
                        a=src.read(k,masked=True).astype("float32")
                        masked_array=np.ma.filled(a,-9999.0)
                        if same_grid:
                            result=masked_array
                        else:
                            result=np.full((profile["height"],profile["width"]),-9999.,dtype="float32")
                            is_count=(k % 11 == 0)
                            reproject(source=masked_array,destination=result,
                                src_transform=src.transform,src_crs=src.crs,src_nodata=-9999.,
                                dst_transform=profile["transform"],dst_crs=profile["crs"],dst_nodata=-9999.,
                                resampling=Resampling.nearest if is_count else Resampling.bilinear,
                                init_dest_nodata=True,num_threads=2)
                        dst.write(result,k)
                        dst.set_band_description(k,f"Q{(k-1)//11+1}_"+
                            (BANDS[(k-1)%11] if (k-1)%11<10 else "nclear"))
            rows=audit_raster(aligned,profile,ROI_PATH)
            if target.exists(): raise FileExistsError("Target appeared during export")
            shutil.copy2(aligned,target)
            audit_raster(target,profile,ROI_PATH)
            records += [dict(site=site,years=f"{start}-{end}",status="exported",**r) for r in rows]
            print("VERIFIED SAVED",target,flush=True)
    except Exception as exc:
        print("FAILED",site,repr(exc),flush=True)
        traceback.print_exc(limit=2)
        records.append(dict(site=site,years=f"{start}-{end}",status="failed",error=str(exc)))
    pd.DataFrame(records).to_csv(OUTPUT_DIR/"quarterly_qa.csv",index=False)
print("QA table:",OUTPUT_DIR/"quarterly_qa.csv")


## Interpretation and safeguards

Quarterly count bands are observation counts, not normalized reflectances. A missing ROI path leaves ROI coverage as NaN; set `ROI_PATH` before running for the requested ROI audit. Downloaded grids are compared with canonical grids; discrepancies are printed and resampled explicitly (bilinear spectral bands, nearest clear counts) if footprint overlap is at least 98%. Files failing QA are preserved and explicitly reported; move or rename them manually before a replacement export. Three-year windows can contain imagery after UAV acquisition. The full-mosaic domain remains unchanged. A subsequent focused LOSO notebook will compare the four specified predictor combinations at 75/100/125 m.